# Tema 5 - Componentes Dinámicos

**Teoría de Circuitos - ETSI, Universidad de Sevilla**

---

## Objetivos de aprendizaje

- Comprender la relación constitutiva del condensador ($q = Cu$, $i = C\frac{du}{dt}$) y sus propiedades de almacenamiento de energía
- Comprender la relación constitutiva de la bobina ($\phi = Li$, $u = L\frac{di}{dt}$) y sus propiedades de almacenamiento de energía
- Calcular combinaciones serie y paralelo de condensadores e inductancias
- Entender el acoplamiento magnético, la inductancia mutua y el transformador ideal
- Reconocer la dualidad C $\leftrightarrow$ L y aplicarla para simplificar análisis
- Conocer los modelos de componentes pasivos reales (parásitos)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
import schemdraw
import schemdraw.elements as elm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA = '#cb181d'       # rojo - limites, destacados
COLOR_PUNTO = '#238b45'       # verde - puntos de operacion, resultados
COLOR_AUX = '#a6cee3'         # azul claro - curvas auxiliares
COLOR_AUX2 = '#b2df8a'        # verde claro - curvas auxiliares 2

print('Configuracion lista.')

---

## 1. El condensador

### 1.1 Introduccion e intuicion

El **condensador** es un componente que almacena energia en forma de **campo electrico**. Fisicamente, consiste en dos placas conductoras separadas por un material dielectrico.

**Analogia hidraulica**: un condensador se comporta como un deposito de agua flexible (membrana). La presion (tension) depende de cuanta agua (carga) se ha acumulado, y el flujo (corriente) solo existe mientras la membrana se esta deformando (la tension cambia).

### 1.2 Relacion constitutiva

La carga almacenada es proporcional a la tension entre sus terminales:

$$\boxed{q = C \cdot u}$$

donde $C$ es la **capacidad** en faradios (F). Derivando respecto al tiempo:

$$\boxed{i = C \frac{du}{dt}}$$

Y la tension a partir de la corriente (forma integral):

$$\boxed{u(t) = \frac{1}{C} \int_{t_0}^{t} i(\tau)\,d\tau + u(t_0)}$$

| Formula | Descripcion |
|---------|-------------|
| $q = C \cdot u$ | Carga almacenada |
| $i = C \dfrac{du}{dt}$ | Corriente en funcion de la derivada de tension |
| $u(t) = \dfrac{1}{C}\int i\,dt + u(t_0)$ | Tension a partir de la corriente |
| $w_C = \dfrac{1}{2}C u^2 \geq 0$ | **Energia almacenada** (siempre positiva) |

### 1.3 Propiedades fundamentales

- **Elemento almacenador de energia**: $w_C = \frac{1}{2}Cu^2 \geq 0$. El condensador **no disipa** energia, la almacena y la devuelve.
- **La tension NO puede cambiar instantaneamente**: $u(0^-) = u(0^+)$. Un cambio instantaneo requeriria corriente infinita ($i = C \frac{du}{dt} \to \infty$).
- En **corriente continua** (regimen permanente), $\frac{du}{dt} = 0 \implies i = 0$: el condensador se comporta como un **circuito abierto**.

> **Error frecuente**: confundir la condicion de continuidad. Es la **tension** del condensador la que no puede saltar, no la corriente.

In [ ]:
# Simbolo del condensador con schemdraw
fig, ax = plt.subplots(figsize=(7, 4))
ax.set_title('Condensador: simbolo y convenio de signos', fontsize=14, fontweight='bold')
d = schemdraw.Drawing(canvas=ax)
d += elm.Line().right().length(2)
d += elm.Capacitor().right().label(r'$C$', loc='top').label(r'$u$', loc='bottom')
d += elm.Line().right().length(2)
d += elm.CurrentLabelInline(direction='in').at(d.elements[0].end).label(r'$i$')
d.draw()
plt.tight_layout()
plt.show()

In [ ]:
# Relacion lineal i = C * du/dt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: i vs du/dt para distintos C
dudt = np.linspace(-5, 5, 100)
for C_val, label_c, color in zip([1e-6, 10e-6, 100e-6],
                                  [r'$C = 1\;\mu$F', r'$C = 10\;\mu$F', r'$C = 100\;\mu$F'],
                                  [COLOR_AUX, COLOR_PRINCIPAL, COLOR_RECTA]):
    axes[0].plot(dudt, C_val * dudt * 1e3, color=color, lw=2, label=label_c)
axes[0].set_xlabel(r'$du/dt$ (V/s)')
axes[0].set_ylabel(r'$i$ (mA)')
axes[0].set_title(r'Corriente vs tasa de cambio de tension')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', lw=0.5)
axes[0].axvline(x=0, color='k', lw=0.5)

# Derecha: formas de onda u(t) e i(t) para entrada en escalon de corriente
t = np.linspace(0, 5, 500)
C = 1e-3  # 1 mF
I0 = 2e-3  # 2 mA
u0 = 1.0  # V
# Corriente constante de 0 a 3s, luego 0
i_t = np.where((t >= 1) & (t <= 4), I0, 0)
u_t = np.zeros_like(t)
dt_step = t[1] - t[0]
u_t[0] = u0
for k in range(1, len(t)):
    u_t[k] = u_t[k-1] + i_t[k-1] * dt_step / C

ax2 = axes[1]
ax2.plot(t, u_t, color=COLOR_PRINCIPAL, lw=2.5, label=r'$u_C(t)$ (V)')
ax2_twin = ax2.twinx()
ax2_twin.plot(t, i_t * 1e3, color=COLOR_RECTA, lw=2, ls='--', label=r'$i(t)$ (mA)')
ax2.set_xlabel(r'Tiempo (s)')
ax2.set_ylabel(r'$u_C$ (V)', color=COLOR_PRINCIPAL)
ax2_twin.set_ylabel(r'$i$ (mA)', color=COLOR_RECTA)
ax2.set_title('Tension del condensador ante pulso de corriente')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=11, loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 2. Asociacion de condensadores

### 2.1 Condensadores en serie

Cuando $n$ condensadores se conectan en **serie**, la capacidad equivalente es:

$$\boxed{\frac{1}{C_{eq}} = \sum_{k=1}^{n} \frac{1}{C_k}}$$

Para dos condensadores en serie:

$$C_{eq} = \frac{C_1 \cdot C_2}{C_1 + C_2}$$

### 2.2 Condensadores en paralelo

Cuando $n$ condensadores se conectan en **paralelo**, la capacidad equivalente es:

$$\boxed{C_{eq} = \sum_{k=1}^{n} C_k}$$

> **Atencion**: las reglas de asociacion de condensadores son **opuestas** a las de resistencias. Serie $\to$ producto/suma (como resistencias en paralelo). Paralelo $\to$ suma directa (como resistencias en serie).

| Configuracion | Formula | Analogia con resistencias |
|---------------|---------|--------------------------|
| **Serie** | $\dfrac{1}{C_{eq}} = \sum \dfrac{1}{C_k}$ | Como resistencias en **paralelo** |
| **Paralelo** | $C_{eq} = \sum C_k$ | Como resistencias en **serie** |

In [ ]:
# Asociacion serie y paralelo de condensadores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Serie
axes[0].set_title('Condensadores en serie', fontsize=14, fontweight='bold')
d1 = schemdraw.Drawing(canvas=axes[0])
d1 += elm.Line().right().length(1)
d1 += elm.Capacitor().right().label(r'$C_1$', loc='top')
d1 += elm.Capacitor().right().label(r'$C_2$', loc='top')
d1 += elm.Capacitor().right().label(r'$C_3$', loc='top')
d1 += elm.Line().right().length(1)
d1.draw()

# Paralelo
axes[1].set_title('Condensadores en paralelo', fontsize=14, fontweight='bold')
d2 = schemdraw.Drawing(canvas=axes[1])
d2 += elm.Line().right().length(2)
d2 += (j1 := elm.Dot())
d2 += elm.Line().right().length(1)
d2 += (j2 := elm.Dot())
d2 += elm.Line().right().length(1)
d2 += (j3 := elm.Dot())
d2 += elm.Line().right().length(2)
d2 += (j4 := elm.Dot())
d2 += elm.Capacitor().at(j1.center).down().label(r'$C_1$', loc='left')
d2 += (b1 := elm.Dot())
d2 += elm.Capacitor().at(j2.center).down().label(r'$C_2$', loc='left')
d2 += (b2 := elm.Dot())
d2 += elm.Capacitor().at(j3.center).down().label(r'$C_3$', loc='left')
d2 += (b3 := elm.Dot())
d2 += elm.Line().at(b1.center).right().tox(b3.center)
d2 += elm.Line().at(j4.center).down().toy(b3.center)
d2.draw()

plt.tight_layout()
plt.show()

---

## 3. La inductancia (bobina)

### 3.1 Introduccion e intuicion

La **inductancia** (bobina) almacena energia en forma de **campo magnetico**. Fisicamente, es un conductor enrollado (solenoide) que genera un flujo magnetico proporcional a la corriente.

**Analogia hidraulica**: una bobina se comporta como una **turbina con inercia**. Una vez que el agua (corriente) fluye, la turbina se resiste a detenerse; y si esta parada, se resiste a arrancar.

### 3.2 Relacion constitutiva

El flujo magnetico concatenado es proporcional a la corriente:

$$\boxed{\phi = L \cdot i}$$

donde $L$ es la **inductancia** en henrios (H). Derivando (ley de Faraday):

$$\boxed{u = L \frac{di}{dt}}$$

Y la corriente a partir de la tension (forma integral):

$$\boxed{i(t) = \frac{1}{L} \int_{t_0}^{t} u(\tau)\,d\tau + i(t_0)}$$

| Formula | Descripcion |
|---------|-------------|
| $\phi = L \cdot i$ | Flujo magnetico concatenado |
| $u = L \dfrac{di}{dt}$ | Tension en funcion de la derivada de corriente |
| $i(t) = \dfrac{1}{L}\int u\,dt + i(t_0)$ | Corriente a partir de la tension |
| $w_L = \dfrac{1}{2}L i^2 \geq 0$ | **Energia almacenada** (siempre positiva) |

### 3.3 Propiedades fundamentales

- **Elemento almacenador de energia**: $w_L = \frac{1}{2}Li^2 \geq 0$. La bobina **no disipa** energia.
- **La corriente NO puede cambiar instantaneamente**: $i(0^-) = i(0^+)$. Un cambio instantaneo requeriria tension infinita ($u = L \frac{di}{dt} \to \infty$).
- En **corriente continua** (regimen permanente), $\frac{di}{dt} = 0 \implies u = 0$: la bobina se comporta como un **cortocircuito**.

> **Error frecuente**: confundir la condicion de continuidad. Es la **corriente** de la bobina la que no puede saltar, no la tension.

In [ ]:
# Simbolo de la bobina con schemdraw
fig, ax = plt.subplots(figsize=(7, 4))
ax.set_title('Inductancia: simbolo y convenio de signos', fontsize=14, fontweight='bold')
d = schemdraw.Drawing(canvas=ax)
d += elm.Line().right().length(2)
d += elm.Inductor2().right().label(r'$L$', loc='top').label(r'$u$', loc='bottom')
d += elm.Line().right().length(2)
d += elm.CurrentLabelInline(direction='in').at(d.elements[0].end).label(r'$i$')
d.draw()
plt.tight_layout()
plt.show()

In [ ]:
# Formas de onda de la bobina ante pulso de tension
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: u vs di/dt para distintos L
didt = np.linspace(-5, 5, 100)
for L_val, label_l, color in zip([1e-3, 10e-3, 100e-3],
                                  [r'$L = 1$ mH', r'$L = 10$ mH', r'$L = 100$ mH'],
                                  [COLOR_AUX, COLOR_PRINCIPAL, COLOR_RECTA]):
    axes[0].plot(didt, L_val * didt, color=color, lw=2, label=label_l)
axes[0].set_xlabel(r'$di/dt$ (A/s)')
axes[0].set_ylabel(r'$u$ (V)')
axes[0].set_title(r'Tension vs tasa de cambio de corriente')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', lw=0.5)
axes[0].axvline(x=0, color='k', lw=0.5)

# Derecha: formas de onda i(t) y u(t) para pulso de tension
t = np.linspace(0, 5, 500)
L = 1.0  # 1 H
U0 = 2.0  # 2 V
i0 = 0.5  # A
# Tension constante de 1 a 4s, luego 0
u_t = np.where((t >= 1) & (t <= 4), U0, 0)
i_t = np.zeros_like(t)
dt_step = t[1] - t[0]
i_t[0] = i0
for k in range(1, len(t)):
    i_t[k] = i_t[k-1] + u_t[k-1] * dt_step / L

ax2 = axes[1]
ax2.plot(t, i_t, color=COLOR_PRINCIPAL, lw=2.5, label=r'$i_L(t)$ (A)')
ax2_twin = ax2.twinx()
ax2_twin.plot(t, u_t, color=COLOR_RECTA, lw=2, ls='--', label=r'$u(t)$ (V)')
ax2.set_xlabel(r'Tiempo (s)')
ax2.set_ylabel(r'$i_L$ (A)', color=COLOR_PRINCIPAL)
ax2_twin.set_ylabel(r'$u$ (V)', color=COLOR_RECTA)
ax2.set_title('Corriente de la bobina ante pulso de tension')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=11, loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 4. Asociacion de inductancias

### 4.1 Inductancias en serie

Cuando $n$ bobinas se conectan en **serie** (sin acoplamiento magnetico):

$$\boxed{L_{eq} = \sum_{k=1}^{n} L_k}$$

### 4.2 Inductancias en paralelo

Cuando $n$ bobinas se conectan en **paralelo**:

$$\boxed{\frac{1}{L_{eq}} = \sum_{k=1}^{n} \frac{1}{L_k}}$$

Para dos bobinas en paralelo:

$$L_{eq} = \frac{L_1 \cdot L_2}{L_1 + L_2}$$

> **Nota**: las reglas de asociacion de inductancias son **iguales** a las de resistencias. Serie $\to$ suma directa. Paralelo $\to$ producto/suma.

| Configuracion | Formula | Analogia con resistencias |
|---------------|---------|--------------------------|
| **Serie** | $L_{eq} = \sum L_k$ | Como resistencias en **serie** |
| **Paralelo** | $\dfrac{1}{L_{eq}} = \sum \dfrac{1}{L_k}$ | Como resistencias en **paralelo** |

---

## 5. Energia almacenada en condensadores e inductancias

In [ ]:
# Comparacion de energia almacenada en C y L
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Energia en condensador: w = 0.5*C*u^2
u_vals = np.linspace(0, 10, 200)
for C_val, label_c, color in zip([10e-6, 100e-6, 1e-3],
                                  [r'$C = 10\;\mu$F', r'$C = 100\;\mu$F', r'$C = 1$ mF'],
                                  [COLOR_AUX, COLOR_PRINCIPAL, COLOR_RECTA]):
    w_c = 0.5 * C_val * u_vals**2 * 1e3  # en mJ
    axes[0].plot(u_vals, w_c, color=color, lw=2, label=label_c)
axes[0].set_xlabel(r'$u_C$ (V)')
axes[0].set_ylabel(r'$w_C$ (mJ)')
axes[0].set_title(r'Energia almacenada en condensador: $w_C = \frac{1}{2}Cu^2$')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Energia en bobina: w = 0.5*L*i^2
i_vals = np.linspace(0, 5, 200)
for L_val, label_l, color in zip([1e-3, 10e-3, 100e-3],
                                  [r'$L = 1$ mH', r'$L = 10$ mH', r'$L = 100$ mH'],
                                  [COLOR_AUX, COLOR_PRINCIPAL, COLOR_RECTA]):
    w_l = 0.5 * L_val * i_vals**2 * 1e3  # en mJ
    axes[1].plot(i_vals, w_l, color=color, lw=2, label=label_l)
axes[1].set_xlabel(r'$i_L$ (A)')
axes[1].set_ylabel(r'$w_L$ (mJ)')
axes[1].set_title(r'Energia almacenada en bobina: $w_L = \frac{1}{2}Li^2$')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6. Acoplamiento magnetico

### 6.1 Inductancia mutua

Cuando dos bobinas estan proximas, parte del flujo magnetico de una atraviesa a la otra. Se define la **inductancia mutua** $M$:

$$\boxed{k = \frac{M}{\sqrt{L_1 \cdot L_2}}, \qquad 0 \leq k \leq 1}$$

donde $k$ es el **coeficiente de acoplamiento**:
- $k = 0$: sin acoplamiento (bobinas muy separadas)
- $k = 1$: acoplamiento perfecto (todo el flujo es compartido, transformador ideal)
- $0 < k < 1$: acoplamiento parcial (caso real)

### 6.2 Ecuaciones del transformador

Las ecuaciones constitutivas de dos bobinas acopladas son:

$$\boxed{u_1 = L_1 \frac{di_1}{dt} \pm M \frac{di_2}{dt}}$$

$$\boxed{u_2 = \pm M \frac{di_1}{dt} + L_2 \frac{di_2}{dt}}$$

El signo $\pm$ depende del **convenio de puntos** (*dot convention*):
- **Signo +**: si ambas corrientes **entran** por los terminales con punto, o ambas **salen**
- **Signo -**: si una corriente entra por el punto y la otra sale por el punto

### 6.3 Transformador ideal

Un transformador ideal ($k = 1$, sin perdidas) cumple:

$$\boxed{\frac{u_1}{u_2} = \frac{N_1}{N_2} = n} \qquad \boxed{i_1 \cdot N_1 = i_2 \cdot N_2}$$

donde $n = N_1/N_2$ es la **relacion de transformacion**.

| Propiedad | Formula |
|-----------|---------|
| Relacion de tensiones | $u_1 / u_2 = N_1 / N_2$ |
| Relacion de corrientes | $i_1 \cdot N_1 = i_2 \cdot N_2$ |
| Impedancia reflejada | $Z_{\text{ref}} = n^2 \cdot Z_2$ |
| Potencia | $p_1 = p_2$ (sin perdidas) |

In [ ]:
# Simbolo del transformador con convenio de puntos
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_title('Transformador: simbolo con convenio de puntos', fontsize=14, fontweight='bold')
d = schemdraw.Drawing(canvas=ax)
d += (xf := elm.Transformer(t1=4, t2=4).label(r'$M$', loc='center'))
d += elm.Dot().at(xf.p1).label(r'$\bullet$', loc='left')
d += elm.Line().at(xf.p1).left().length(1.5).label(r'$u_1$', loc='bottom')
d += elm.Line().at(xf.p2).left().length(1.5)
d += elm.Line().at(xf.s1).right().length(1.5).label(r'$u_2$', loc='bottom')
d += elm.Dot().at(xf.s1).label(r'$\bullet$', loc='right')
d += elm.Line().at(xf.s2).right().length(1.5)
d += elm.CurrentLabelInline(direction='in').at(xf.p1).label(r'$i_1$')
d += elm.CurrentLabelInline(direction='in').at(xf.s1).label(r'$i_2$')
d.draw()
plt.tight_layout()
plt.show()

---

## 7. Tabla de dualidad C $\leftrightarrow$ L

El condensador y la inductancia son **elementos duales**. Cada propiedad de uno tiene su correspondiente en el otro, intercambiando variables:

| Condensador (C) | Inductancia (L) |
|-----------------|-----------------|
| $q = Cu$ | $\phi = Li$ |
| $i = C\dfrac{du}{dt}$ | $u = L\dfrac{di}{dt}$ |
| $u = \dfrac{1}{C}\int i\,dt + u(t_0)$ | $i = \dfrac{1}{L}\int u\,dt + i(t_0)$ |
| $w_C = \dfrac{1}{2}Cu^2$ | $w_L = \dfrac{1}{2}Li^2$ |
| $u(0^-) = u(0^+)$ | $i(0^-) = i(0^+)$ |
| CC: circuito **abierto** | CC: **cortocircuito** |
| Serie: $\dfrac{1}{C_{eq}} = \sum\dfrac{1}{C_k}$ | Serie: $L_{eq} = \sum L_k$ |
| Paralelo: $C_{eq} = \sum C_k$ | Paralelo: $\dfrac{1}{L_{eq}} = \sum\dfrac{1}{L_k}$ |
| Variable de estado: $u$ (tension) | Variable de estado: $i$ (corriente) |

> **Truco para el examen**: si sabes resolver un circuito con condensadores, puedes resolver el dual con bobinas simplemente intercambiando $u \leftrightarrow i$, serie $\leftrightarrow$ paralelo, $C \leftrightarrow L$.

---

## 8. Elementos pasivos reales

Los componentes reales no son ideales. Cada uno tiene **elementos parasitos** que afectan su comportamiento, especialmente a alta frecuencia.

### 8.1 Resistencia real

Una resistencia real tiene:
- **Inductancia parasita** $L_p$ (debida al cuerpo del componente y a los terminales)
- **Capacidad parasita** $C_p$ (entre terminales)

A baja frecuencia se comporta como resistencia ideal; a alta frecuencia, los parasitos dominan.

### 8.2 Condensador real

Un condensador real tiene:
- **ESR** (*Equivalent Series Resistance*): resistencia del dielectrico y las placas
- **ESL** (*Equivalent Series Inductance*): inductancia de los terminales

$$Z_C = \text{ESR} + j\omega \cdot \text{ESL} + \frac{1}{j\omega C}$$

Existe una **frecuencia de autorresonancia** donde $Z_C$ es minima.

### 8.3 Bobina real

Una bobina real tiene:
- **Resistencia del devanado** $R_w$: perdidas ohmicas en el hilo
- **Capacidad entre espiras** $C_p$: capacidad parasita entre vueltas del bobinado

A baja frecuencia domina $R_w + j\omega L$; a alta frecuencia, $C_p$ crea una resonancia parasita.

| Componente | Modelo ideal | Parasitos principales |
|------------|-------------|----------------------|
| Resistencia | $R$ | $L_p$ (serie), $C_p$ (paralelo) |
| Condensador | $C$ | ESR (serie), ESL (serie) |
| Bobina | $L$ | $R_w$ (serie), $C_p$ (paralelo) |

In [ ]:
# Modelos de componentes pasivos reales
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Resistencia real
axes[0].set_title('Resistencia real', fontsize=13, fontweight='bold')
d1 = schemdraw.Drawing(canvas=axes[0])
d1 += elm.Line().right().length(1)
d1 += elm.Resistor().right().label(r'$R$', loc='top')
d1 += elm.Inductor2().right().label(r'$L_p$', loc='top')
d1 += elm.Line().right().length(1)
d1.draw()

# Condensador real
axes[1].set_title('Condensador real', fontsize=13, fontweight='bold')
d2 = schemdraw.Drawing(canvas=axes[1])
d2 += elm.Line().right().length(1)
d2 += elm.Resistor().right().label(r'ESR', loc='top')
d2 += elm.Inductor2().right().label(r'ESL', loc='top')
d2 += elm.Capacitor().right().label(r'$C$', loc='top')
d2 += elm.Line().right().length(1)
d2.draw()

# Bobina real
axes[2].set_title('Bobina real', fontsize=13, fontweight='bold')
d3 = schemdraw.Drawing(canvas=axes[2])
d3 += elm.Line().right().length(1)
d3 += elm.Resistor().right().label(r'$R_w$', loc='top')
d3 += elm.Inductor2().right().label(r'$L$', loc='top')
d3 += elm.Line().right().length(1)
d3.draw()

plt.tight_layout()
plt.show()

---

## 9. Metodologia de resolucion

### Pasos para resolver problemas con componentes dinamicos:

**Paso 1:** Identificar todos los condensadores e inductancias del circuito y sus condiciones iniciales $u_C(t_0)$, $i_L(t_0)$.

**Paso 2:** Aplicar las condiciones de continuidad en $t = 0$:
- $u_C(0^+) = u_C(0^-)$ para cada condensador
- $i_L(0^+) = i_L(0^-)$ para cada bobina

**Paso 3:** Plantear las ecuaciones constitutivas ($i = C\frac{du}{dt}$, $u = L\frac{di}{dt}$) junto con las leyes de Kirchhoff.

**Paso 4:** Para asociaciones, calcular $C_{eq}$ o $L_{eq}$ antes de resolver.

**Paso 5:** Calcular la energia almacenada si se pide: $w_C = \frac{1}{2}Cu^2$, $w_L = \frac{1}{2}Li^2$.

**Paso 6:** Para transformadores, aplicar las relaciones $u_1/u_2 = N_1/N_2$ e $i_1 N_1 = i_2 N_2$.

---

## 10. Ejemplos resueltos

### 10.1 Ejemplo A: Carga y energia en un condensador

#### Ejercicio resuelto: Condensador con corriente conocida

**Datos:** $C = 10\;\mu$F, $u_C(0) = 2$ V. La corriente es $i(t) = 5$ mA constante durante $0 \leq t \leq 4$ ms.

**Paso 1:** Calcular $u_C(t)$ para $0 \leq t \leq 4$ ms.

$$u_C(t) = \frac{1}{C}\int_0^t i\,d\tau + u_C(0) = \frac{1}{10\;\mu\text{F}} \cdot 5\;\text{mA} \cdot t + 2$$

$$u_C(t) = \frac{5 \times 10^{-3}}{10 \times 10^{-6}} \cdot t + 2 = 500t + 2 \;\text{V}$$

**Paso 2:** En $t = 4$ ms:

$$u_C(4\;\text{ms}) = 500 \times 4 \times 10^{-3} + 2 = 2 + 2 = 4\;\text{V}$$

**Paso 3:** Carga almacenada en $t = 4$ ms:

$$q = C \cdot u_C = 10\;\mu\text{F} \times 4 = 40\;\mu\text{C}$$

**Paso 4:** Energia almacenada en $t = 4$ ms:

$$\boxed{w_C = \frac{1}{2}Cu_C^2 = \frac{1}{2} \times 10\;\mu\text{F} \times 4^2 = 80\;\mu\text{J}}$$

### 10.2 Ejemplo B: Flujo y energia en una bobina

#### Ejercicio resuelto: Bobina con tension conocida

**Datos:** $L = 50$ mH, $i_L(0) = 0$ A. La tension es $u(t) = 10$ V constante durante $0 \leq t \leq 2$ ms.

**Paso 1:** Calcular $i_L(t)$:

$$i_L(t) = \frac{1}{L}\int_0^t u\,d\tau + i_L(0) = \frac{10}{50\;\text{mH}} \cdot t + 0 = 200t \;\text{A}$$

**Paso 2:** En $t = 2$ ms:

$$i_L(2\;\text{ms}) = 200 \times 2 \times 10^{-3} = 0.4\;\text{A}$$

**Paso 3:** Flujo magnetico:

$$\phi = L \cdot i_L = 50\;\text{mH} \times 0.4 = 20\;\text{mWb}$$

**Paso 4:** Energia almacenada:

$$\boxed{w_L = \frac{1}{2}Li_L^2 = \frac{1}{2} \times 50\;\text{mH} \times 0.4^2 = 4\;\text{mJ}}$$

---

## 11. Catalogo completo de ejercicios: todos los patrones

Esta seccion clasifica todos los tipos de problemas que pueden aparecer en examen sobre componentes dinamicos.

| # | Tipo | Componente | Ecuacion clave | Dificultad |
|---|------|-----------|----------------|------------|
| 1 | Tension del condensador a partir de forma de onda de corriente | C | $u = \frac{1}{C}\int i\,dt + u(t_0)$ | Media |
| 2 | Corriente de la bobina a partir de forma de onda de tension | L | $i = \frac{1}{L}\int u\,dt + i(t_0)$ | Media |
| 3 | Energia almacenada en C o L | C, L | $w = \frac{1}{2}Cu^2$, $w = \frac{1}{2}Li^2$ | Baja |
| 4 | Condensadores en serie | C | $1/C_{eq} = \sum 1/C_k$ | Baja |
| 5 | Condensadores en paralelo | C | $C_{eq} = \sum C_k$ | Baja |
| 6 | Inductancias en serie | L | $L_{eq} = \sum L_k$ | Baja |
| 7 | Inductancias en paralelo | L | $1/L_{eq} = \sum 1/L_k$ | Baja |
| 8 | Condiciones de continuidad: $u_C(0^+)$, $i_L(0^+)$ | C, L | Continuidad | Media |
| 9 | Bobinas acopladas (inductancia mutua) | L, M | $u = L\frac{di}{dt} \pm M\frac{di}{dt}$ | Alta |
| 10 | Transformador ideal | Trafo | $u_1/u_2 = N_1/N_2$ | Media |

### 11.1 Tipo 1: Tension del condensador a partir de forma de onda de corriente

Se da $i(t)$ como funcion a tramos y se pide $u_C(t)$. Se aplica:

$$\boxed{u_C(t) = \frac{1}{C}\int_{t_0}^{t} i(\tau)\,d\tau + u_C(t_0)}$$

**Como afectan los parametros:**
- Si **$C$ aumenta** $\to$ la tension cambia mas lentamente (mas inercia)
- Si **$i$ aumenta** $\to$ la tension cambia mas rapido
- Si **$u_C(t_0)$ cambia** $\to$ solo desplaza la curva verticalmente

#### Ejercicio resuelto: Corriente triangular en condensador

**Datos:** $C = 100\;\mu$F, $u_C(0) = 0$ V. La corriente es triangular: sube linealmente de 0 a 10 mA en $0 \leq t \leq 1$ ms, luego baja linealmente de 10 mA a 0 en $1 \leq t \leq 2$ ms.

**Tramo 1** ($0 \leq t \leq 1$ ms): $i(t) = 10t$ A (con $t$ en segundos, $i$ en A: $i = 10 \times 10^3 \times 10^{-3} \cdot t = 10t$).

En realidad, si sube de 0 a 10 mA en 1 ms: $i(t) = \frac{10 \times 10^{-3}}{1 \times 10^{-3}} t = 10t$ A.

$$u_C(t) = \frac{1}{100\;\mu\text{F}}\int_0^t 10\tau\,d\tau = \frac{10}{100 \times 10^{-6}} \cdot \frac{t^2}{2} = 50000 \cdot t^2 \;\text{V}$$

En $t = 1$ ms: $u_C = 50000 \times (10^{-3})^2 = 0.05$ V.

**Tramo 2** ($1 \leq t \leq 2$ ms): $i(t) = 10 \times 10^{-3} - 10(t - 10^{-3})$ A. Se integra analogamente partiendo de $u_C(1\;\text{ms}) = 0.05$ V.

$$\boxed{u_C(2\;\text{ms}) = 0.1\;\text{V}}$$

In [ ]:
# Tipo 1: Tension del condensador a partir de corriente triangular
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

C = 100e-6  # 100 uF
t = np.linspace(0, 2e-3, 1000)
# Corriente triangular
i_t = np.where(t <= 1e-3, 10 * t, 10e-3 - 10 * (t - 1e-3))

# Tension por integracion numerica
u_t = np.zeros_like(t)
dt_step = t[1] - t[0]
for k in range(1, len(t)):
    u_t[k] = u_t[k-1] + i_t[k-1] * dt_step / C

axes[0].plot(t * 1e3, i_t * 1e3, color=COLOR_RECTA, lw=2.5)
axes[0].set_xlabel(r'Tiempo (ms)')
axes[0].set_ylabel(r'$i(t)$ (mA)')
axes[0].set_title('Corriente triangular aplicada')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t * 1e3, u_t, color=COLOR_PRINCIPAL, lw=2.5)
axes[1].set_xlabel(r'Tiempo (ms)')
axes[1].set_ylabel(r'$u_C(t)$ (V)')
axes[1].set_title(r'Tension resultante en el condensador')
axes[1].grid(True, alpha=0.3)
axes[1].annotate(f'{u_t[-1]:.3f} V',
                 xy=(t[-1]*1e3, u_t[-1]),
                 xytext=(t[-1]*1e3 - 0.5, u_t[-1] + 0.01),
                 fontsize=12, color=COLOR_PUNTO, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2))

plt.tight_layout()
plt.show()

### 11.2 Tipo 2: Corriente de la bobina a partir de forma de onda de tension

Se da $u(t)$ como funcion a tramos y se pide $i_L(t)$. Se aplica:

$$\boxed{i_L(t) = \frac{1}{L}\int_{t_0}^{t} u(\tau)\,d\tau + i_L(t_0)}$$

**Como afectan los parametros:**
- Si **$L$ aumenta** $\to$ la corriente cambia mas lentamente (mas inercia)
- Si **$u$ aumenta** $\to$ la corriente cambia mas rapido

#### Ejercicio resuelto: Pulso rectangular de tension en bobina

**Datos:** $L = 20$ mH, $i_L(0) = 1$ A. La tension es $u = 4$ V durante $0 \leq t \leq 5$ ms, luego $u = 0$.

**Paso 1:** Para $0 \leq t \leq 5$ ms:

$$i_L(t) = \frac{1}{20\;\text{mH}}\int_0^t 4\,d\tau + 1 = \frac{4}{0.02}t + 1 = 200t + 1\;\text{A}$$

**Paso 2:** En $t = 5$ ms:

$$i_L(5\;\text{ms}) = 200 \times 5 \times 10^{-3} + 1 = 2\;\text{A}$$

**Paso 3:** Para $t > 5$ ms, $u = 0 \implies di_L/dt = 0$, la corriente se mantiene constante:

$$\boxed{i_L(t) = 2\;\text{A} \quad \text{para } t > 5\;\text{ms}}$$

In [ ]:
# Tipo 2: Corriente de la bobina ante pulso rectangular de tension
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

L = 20e-3  # 20 mH
i0 = 1.0   # A
t = np.linspace(0, 10e-3, 1000)
u_t = np.where(t <= 5e-3, 4.0, 0.0)

i_t = np.zeros_like(t)
dt_step = t[1] - t[0]
i_t[0] = i0
for k in range(1, len(t)):
    i_t[k] = i_t[k-1] + u_t[k-1] * dt_step / L

axes[0].plot(t * 1e3, u_t, color=COLOR_RECTA, lw=2.5)
axes[0].set_xlabel(r'Tiempo (ms)')
axes[0].set_ylabel(r'$u(t)$ (V)')
axes[0].set_title('Tension aplicada (pulso)')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.5, 5)

axes[1].plot(t * 1e3, i_t, color=COLOR_PRINCIPAL, lw=2.5)
axes[1].set_xlabel(r'Tiempo (ms)')
axes[1].set_ylabel(r'$i_L(t)$ (A)')
axes[1].set_title(r'Corriente resultante en la bobina')
axes[1].grid(True, alpha=0.3)
axes[1].annotate(f'{i_t[-1]:.2f} A',
                 xy=(t[-1]*1e3, i_t[-1]),
                 xytext=(t[-1]*1e3 - 3, i_t[-1] + 0.15),
                 fontsize=12, color=COLOR_PUNTO, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2))

plt.tight_layout()
plt.show()

### 11.3 Tipo 3: Calculo de energia almacenada

$$\boxed{w_C = \frac{1}{2}Cu_C^2} \qquad \boxed{w_L = \frac{1}{2}Li_L^2}$$

**Variantes comunes:**
- Calcular la energia en un instante dado
- Calcular la variacion de energia $\Delta w = w(t_2) - w(t_1)$
- Calcular la energia suministrada o absorbida en un intervalo

#### Ejercicio resuelto: Variacion de energia

**Datos:** $C = 47\;\mu$F con $u_C$ que cambia de 3 V a 8 V. Bobina $L = 10$ mH con $i_L$ que cambia de 0.5 A a 2 A.

**Condensador:**

$$\Delta w_C = \frac{1}{2}C(u_2^2 - u_1^2) = \frac{1}{2} \times 47\;\mu\text{F} \times (8^2 - 3^2) = \frac{1}{2} \times 47 \times 10^{-6} \times 55 = 1.29\;\text{mJ}$$

**Bobina:**

$$\Delta w_L = \frac{1}{2}L(i_2^2 - i_1^2) = \frac{1}{2} \times 10\;\text{mH} \times (2^2 - 0.5^2) = \frac{1}{2} \times 0.01 \times 3.75 = 18.75\;\text{mJ}$$

$$\boxed{\Delta w_C = 1.29\;\text{mJ}, \quad \Delta w_L = 18.75\;\text{mJ}}$$

### 11.4 Tipo 4: Condensadores en serie

$$\boxed{\frac{1}{C_{eq}} = \frac{1}{C_1} + \frac{1}{C_2} + \cdots + \frac{1}{C_n}}$$

**Caso especial** (2 condensadores): $C_{eq} = \dfrac{C_1 C_2}{C_1 + C_2}$

**Caso especial** ($n$ condensadores iguales): $C_{eq} = C/n$

#### Ejercicio resuelto: Tres condensadores en serie

**Datos:** $C_1 = 10\;\mu$F, $C_2 = 20\;\mu$F, $C_3 = 30\;\mu$F en serie.

$$\frac{1}{C_{eq}} = \frac{1}{10} + \frac{1}{20} + \frac{1}{30} = \frac{6 + 3 + 2}{60} = \frac{11}{60}$$

$$\boxed{C_{eq} = \frac{60}{11} \approx 5.45\;\mu\text{F}}$$

> **Nota**: el resultado en serie siempre es **menor** que el menor de los condensadores individuales.

### 11.5 Tipo 5: Condensadores en paralelo

$$\boxed{C_{eq} = C_1 + C_2 + \cdots + C_n}$$

#### Ejercicio resuelto: Tres condensadores en paralelo

**Datos:** $C_1 = 10\;\mu$F, $C_2 = 20\;\mu$F, $C_3 = 30\;\mu$F en paralelo.

$$\boxed{C_{eq} = 10 + 20 + 30 = 60\;\mu\text{F}}$$

### 11.6 Tipo 6: Inductancias en serie

$$\boxed{L_{eq} = L_1 + L_2 + \cdots + L_n}$$

#### Ejercicio resuelto: Dos bobinas en serie

**Datos:** $L_1 = 100$ mH, $L_2 = 250$ mH en serie (sin acoplamiento).

$$\boxed{L_{eq} = 100 + 250 = 350\;\text{mH}}$$

### 11.7 Tipo 7: Inductancias en paralelo

$$\boxed{\frac{1}{L_{eq}} = \frac{1}{L_1} + \frac{1}{L_2} + \cdots + \frac{1}{L_n}}$$

#### Ejercicio resuelto: Dos bobinas en paralelo

**Datos:** $L_1 = 100$ mH, $L_2 = 250$ mH en paralelo.

$$L_{eq} = \frac{L_1 \cdot L_2}{L_1 + L_2} = \frac{100 \times 250}{100 + 250} = \frac{25000}{350}$$

$$\boxed{L_{eq} \approx 71.43\;\text{mH}}$$

### 11.8 Tipo 8: Condiciones de continuidad

En el instante de una conmutacion ($t = 0$):

$$\boxed{u_C(0^+) = u_C(0^-)} \qquad \boxed{i_L(0^+) = i_L(0^-)}$$

**Procedimiento:**
1. Resolver el circuito para $t < 0$ (regimen permanente anterior)
2. Determinar $u_C(0^-)$ e $i_L(0^-)$
3. Aplicar continuidad: estos valores se mantienen justo despues de la conmutacion
4. Resolver el nuevo circuito en $t = 0^+$ usando esos valores como condiciones iniciales

#### Ejercicio resuelto: Conmutacion con C y L

**Datos:** Circuito con $V_s = 12$ V, $R = 1$ k$\Omega$, $C = 100\;\mu$F. Antes de la conmutacion, el condensador esta cargado a $u_C = 8$ V. En $t = 0$ se abre un interruptor.

**Paso 1:** Condicion inicial: $u_C(0^-) = 8$ V.

**Paso 2:** Continuidad: $u_C(0^+) = u_C(0^-) = 8$ V.

**Paso 3:** La corriente justo despues de la conmutacion depende del nuevo circuito. Si el condensador queda en serie con $R$:

$$i(0^+) = \frac{V_s - u_C(0^+)}{R} = \frac{12 - 8}{1\;\text{k}\Omega} = 4\;\text{mA}$$

$$\boxed{u_C(0^+) = 8\;\text{V}, \quad i(0^+) = 4\;\text{mA}}$$

> **Error frecuente**: asumir que la corriente del condensador tambien es continua. **Solo la tension** del condensador es continua; la corriente puede saltar.

### 11.9 Tipo 9: Bobinas acopladas

$$\boxed{u_1 = L_1 \frac{di_1}{dt} \pm M \frac{di_2}{dt}} \qquad \boxed{u_2 = \pm M \frac{di_1}{dt} + L_2 \frac{di_2}{dt}}$$

**Como afectan los parametros:**
- Si **$M$ aumenta** $\to$ mayor acoplamiento, mas influencia entre bobinas
- Si **$k \to 1$** $\to$ se aproxima al transformador ideal
- El **signo** depende del convenio de puntos (convenio critico en examen)

#### Ejercicio resuelto: Inductancia mutua

**Datos:** $L_1 = 100$ mH, $L_2 = 400$ mH, $k = 0.5$. Las corrientes son $i_1(t) = 2\sin(100t)$ A e $i_2(t) = 0$ A.

**Paso 1:** Calcular $M$:

$$M = k\sqrt{L_1 L_2} = 0.5 \times \sqrt{0.1 \times 0.4} = 0.5 \times 0.2 = 0.1\;\text{H} = 100\;\text{mH}$$

**Paso 2:** Calcular $u_1$ (con signos + por convenio de puntos):

$$u_1 = L_1\frac{di_1}{dt} + M\frac{di_2}{dt} = 0.1 \times 200\cos(100t) + 0 = 20\cos(100t)\;\text{V}$$

**Paso 3:** Calcular $u_2$:

$$u_2 = M\frac{di_1}{dt} + L_2\frac{di_2}{dt} = 0.1 \times 200\cos(100t) + 0 = 20\cos(100t)\;\text{V}$$

$$\boxed{u_1 = 20\cos(100t)\;\text{V}, \quad u_2 = 20\cos(100t)\;\text{V}}$$

### 11.10 Tipo 10: Transformador ideal

$$\boxed{\frac{u_1}{u_2} = \frac{N_1}{N_2} = n} \qquad \boxed{i_1 N_1 = i_2 N_2} \qquad \boxed{Z_{\text{ref}} = n^2 Z_2}$$

**Como afectan los parametros:**
- Si **$n > 1$** $\to$ elevador de tension (tension secundaria menor)
- Si **$n < 1$** $\to$ reductor de tension (tension secundaria mayor)
- La impedancia vista desde el primario se multiplica por $n^2$

#### Ejercicio resuelto: Transformador ideal con carga resistiva

**Datos:** Transformador ideal con $N_1 = 500$ espiras, $N_2 = 100$ espiras. Tension primaria $u_1 = 220$ V (eficaz). Carga $R_L = 10\;\Omega$.

**Paso 1:** Relacion de transformacion:

$$n = \frac{N_1}{N_2} = \frac{500}{100} = 5$$

**Paso 2:** Tension secundaria:

$$u_2 = \frac{u_1}{n} = \frac{220}{5} = 44\;\text{V}$$

**Paso 3:** Corriente secundaria:

$$i_2 = \frac{u_2}{R_L} = \frac{44}{10} = 4.4\;\text{A}$$

**Paso 4:** Corriente primaria:

$$i_1 = \frac{i_2}{n} = \frac{4.4}{5} = 0.88\;\text{A}$$

**Paso 5:** Impedancia reflejada al primario:

$$Z_{\text{ref}} = n^2 \cdot R_L = 25 \times 10 = 250\;\Omega$$

**Verificacion:** $u_1 / Z_{\text{ref}} = 220/250 = 0.88$ A $= i_1$ $\checkmark$

$$\boxed{u_2 = 44\;\text{V}, \quad i_1 = 0.88\;\text{A}, \quad Z_{\text{ref}} = 250\;\Omega}$$

### Tabla resumen de formulas por tipo

| # | Tipo | Formula clave |
|---|------|--------------|
| 1 | $u_C$ desde $i(t)$ | $u_C = \dfrac{1}{C}\int i\,dt + u_C(t_0)$ |
| 2 | $i_L$ desde $u(t)$ | $i_L = \dfrac{1}{L}\int u\,dt + i_L(t_0)$ |
| 3 | Energia | $w_C = \dfrac{1}{2}Cu^2$, $w_L = \dfrac{1}{2}Li^2$ |
| 4 | C serie | $\dfrac{1}{C_{eq}} = \sum\dfrac{1}{C_k}$ |
| 5 | C paralelo | $C_{eq} = \sum C_k$ |
| 6 | L serie | $L_{eq} = \sum L_k$ |
| 7 | L paralelo | $\dfrac{1}{L_{eq}} = \sum\dfrac{1}{L_k}$ |
| 8 | Continuidad | $u_C(0^+) = u_C(0^-)$, $i_L(0^+) = i_L(0^-)$ |
| 9 | Acopladas | $u = L\dfrac{di}{dt} \pm M\dfrac{di}{dt}$ |
| 10 | Trafo ideal | $u_1/u_2 = N_1/N_2$, $i_1 N_1 = i_2 N_2$ |

---

## 12. Resumen y tabla comparativa C vs L

| Propiedad | Condensador ($C$) | Inductancia ($L$) |
|-----------|-------------------|-------------------|
| **Variable almacenada** | Carga $q = Cu$ | Flujo $\phi = Li$ |
| **Relacion constitutiva** | $i = C\dfrac{du}{dt}$ | $u = L\dfrac{di}{dt}$ |
| **Forma integral** | $u = \dfrac{1}{C}\int i\,dt + u(t_0)$ | $i = \dfrac{1}{L}\int u\,dt + i(t_0)$ |
| **Energia** | $w = \dfrac{1}{2}Cu^2$ | $w = \dfrac{1}{2}Li^2$ |
| **Continuidad** | $u_C(0^-) = u_C(0^+)$ | $i_L(0^-) = i_L(0^+)$ |
| **En CC (perm.)** | Circuito abierto ($i = 0$) | Cortocircuito ($u = 0$) |
| **Serie** | $1/C_{eq} = \sum 1/C_k$ | $L_{eq} = \sum L_k$ |
| **Paralelo** | $C_{eq} = \sum C_k$ | $1/L_{eq} = \sum 1/L_k$ |
| **Variable de estado** | Tension $u_C$ | Corriente $i_L$ |
| **Unidad SI** | Faradio (F) | Henrio (H) |

**Formulas clave para el examen:**

| Formula | Uso |
|---------|-----|
| $\boxed{i = C\dfrac{du}{dt}}$ | Corriente en condensador |
| $\boxed{u = L\dfrac{di}{dt}}$ | Tension en bobina |
| $\boxed{w_C = \dfrac{1}{2}Cu^2}$ | Energia en condensador |
| $\boxed{w_L = \dfrac{1}{2}Li^2}$ | Energia en bobina |
| $\boxed{u_C(0^-) = u_C(0^+)}$ | Continuidad de tension en C |
| $\boxed{i_L(0^-) = i_L(0^+)}$ | Continuidad de corriente en L |
| $\boxed{k = M / \sqrt{L_1 L_2}}$ | Coeficiente de acoplamiento |
| $\boxed{u_1/u_2 = N_1/N_2}$ | Transformador ideal (tensiones) |
| $\boxed{i_1 N_1 = i_2 N_2}$ | Transformador ideal (corrientes) |